# SciBERT Contrastive Regression CV5 Ensemble

This notebook reproduces the SciBERT contrastive regression approach and makes it Colab/GitHub-ready.

It trains 5 fold models and saves:

```text
submissions/submission.csv
submissions/submission_scibert_contrastive_regression_cv5.csv
submissions/oof_scibert_contrastive_regression_cv5.csv
submissions/test_pred_scibert_contrastive_regression_cv5.csv
```

The OOF and test prediction files are designed for later ensembling with another model, such as:

```text
SPECTER2 + CORAL
```


## Cell 0 — Colab GitHub Setup

Run this cell on Google Colab. Replace `GITHUB_REPO` with your real GitHub repository URL.


In [ ]:
import os
import subprocess
from pathlib import Path

GITHUB_REPO = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"

REPO_NAME = Path(GITHUB_REPO).stem.replace(".git", "")
PROJECT_ROOT = Path("/content") / REPO_NAME

IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    print("Running on Google Colab.")

    if "YOUR_USERNAME" in GITHUB_REPO or "YOUR_REPO" in GITHUB_REPO:
        raise ValueError("Please replace GITHUB_REPO with your real GitHub repository URL.")

    if not PROJECT_ROOT.exists():
        print("Cloning:", GITHUB_REPO)
        subprocess.run(["git", "clone", GITHUB_REPO, str(PROJECT_ROOT)], check=True)
    else:
        print("Repository already exists. Pulling latest changes...")
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

    os.chdir(PROJECT_ROOT)
    print("Current working directory:", Path.cwd())

    for file_name in ["train.csv", "public_test.csv", "private_test.csv"]:
        p = PROJECT_ROOT / "data" / file_name
        if not p.exists():
            raise FileNotFoundError(f"Missing required file: {p}")

    print("All required data files found.")
else:
    print("Not running on Colab. Skipping GitHub clone.")
    print("Current working directory:", Path.cwd())


## Cell 0.1 — Install Required Packages on Colab

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q numpy pandas scipy scikit-learn torch transformers tqdm
else:
    print("Not running on Colab. Make sure your local environment has the required packages.")


## Cell 1 — Imports, Paths, and Utilities

In [ ]:
import os
import random
import warnings
from pathlib import Path

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
from scipy.optimize import minimize

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

SEED = 42
MODEL_NAME = "allenai/scibert_scivocab_uncased"
EXPERIMENT_NAME = "scibert_contrastive_regression_cv5"

FOLDS = 5
EPOCHS = 10
ALPHA = 0.1
BATCH_SIZE = 32
MAX_LEN = 128

LABEL_COL = "Label"

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent, Path("/content")]

    content_root = Path("/content")
    if content_root.exists():
        for child in content_root.iterdir():
            if child.is_dir():
                candidates.append(child)

    seen = set()
    unique_candidates = []
    for p in candidates:
        p = p.resolve()
        if p not in seen:
            seen.add(p)
            unique_candidates.append(p)

    for p in unique_candidates:
        if (p / "data" / "train.csv").exists():
            return p

    raise FileNotFoundError(
        "Could not find data/train.csv. On Colab, run the GitHub setup cell first."
    )

ROOT = find_project_root()
DATA_DIR = ROOT / "data"
MODEL_DIR = ROOT / "models" / EXPERIMENT_NAME
SUBMISSION_DIR = ROOT / "submissions"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)
print("Model directory:", MODEL_DIR)
print("Submission directory:", SUBMISSION_DIR)


class OrdinalContrastiveLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, Z, labels):
        Z = F.normalize(Z, p=2, dim=1)
        cosine_sim = F.cosine_similarity(
            Z.unsqueeze(1),
            Z.unsqueeze(0),
            dim=2
        )

        d_ij = 1 - cosine_sim
        w_ij = torch.abs(
            labels.unsqueeze(1) - labels.unsqueeze(0)
        ) / 4.0

        attractive_pull = (1 - w_ij) * (d_ij ** 2)
        repulsive_push = w_ij * (torch.clamp(1 - d_ij, min=0) ** 2)

        loss_matrix = attractive_pull + repulsive_push

        mask = ~torch.eye(
            Z.size(0),
            dtype=torch.bool,
            device=Z.device
        )

        return loss_matrix[mask].mean()


class OptimizedRounder:
    def __init__(self):
        self.coef_ = None

    def _kappa_loss(self, coef, X, y):
        X_p = np.copy(X)

        for i, pred in enumerate(X_p):
            if pred < coef[0]:
                X_p[i] = 1
            elif pred < coef[1]:
                X_p[i] = 2
            elif pred < coef[2]:
                X_p[i] = 3
            elif pred < coef[3]:
                X_p[i] = 4
            else:
                X_p[i] = 5

        return -cohen_kappa_score(y, X_p, weights="quadratic")

    def fit(self, X, y):
        loss_partial = lambda coef: self._kappa_loss(coef, X, y)
        initial_coef = [1.5, 2.5, 3.5, 4.5]

        result = minimize(
            loss_partial,
            initial_coef,
            method="nelder-mead",
            options={"maxiter": 2000}
        )

        self.coef_ = np.sort(result["x"])
        return self.coef_

    def predict(self, X, coef=None):
        if coef is None:
            coef = self.coef_

        coef = np.sort(coef)
        X_p = np.copy(X)

        for i, pred in enumerate(X_p):
            if pred < coef[0]:
                X_p[i] = 1
            elif pred < coef[1]:
                X_p[i] = 2
            elif pred < coef[2]:
                X_p[i] = 3
            elif pred < coef[3]:
                X_p[i] = 4
            else:
                X_p[i] = 5

        return X_p


## Cell 2 — Load Data and Build Datasets

This follows the original SciBERT contrastive regression approach:

- title text only
- normalized year
- venue ID embedding
- target label as continuous regression target


In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")

if LABEL_COL not in train_df.columns and "label" in train_df.columns:
    LABEL_COL = "label"

min_year = train_df["year"].min()
max_year = train_df["year"].max()

venue_map = {
    venue: idx
    for idx, venue in enumerate(train_df["venue"].fillna("UNKNOWN").unique())
}

num_venues = len(venue_map)

train_df["year_norm"] = (
    train_df["year"] - min_year
) / (max_year - min_year + 1e-8)

train_df["venue_id"] = (
    train_df["venue"]
    .fillna("UNKNOWN")
    .map(venue_map)
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Train shape:", train_df.shape)
print("Number of venues:", num_venues)
print("Label distribution:")
print(train_df[LABEL_COL].value_counts().sort_index())

print("Train hash:")
print(pd.util.hash_pandas_object(
    train_df[["id", "title", "venue", "year", "authors", "doi", LABEL_COL]],
    index=True
).sum())


class PaperDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

        self.titles = df["title"].fillna("").astype(str).values
        self.years = df["year_norm"].values
        self.venues = df["venue_id"].values
        self.labels = df[LABEL_COL].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.titles[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "year": torch.tensor(self.years[idx], dtype=torch.float),
            "venue": torch.tensor(self.venues[idx], dtype=torch.long),
            "label": torch.tensor(self.labels[idx], dtype=torch.float),
        }


class InferencePaperDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

        self.titles = df["title"].fillna("").astype(str).values
        self.years = df["year_norm"].values
        self.venues = df["venue_id"].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.titles[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "year": torch.tensor(self.years[idx], dtype=torch.float),
            "venue": torch.tensor(self.venues[idx], dtype=torch.long),
        }


## Cell 3 — Model Definition

The model uses:

1. SciBERT CLS embedding
2. normalized year
3. venue embedding
4. regression head
5. projection head for ordinal contrastive learning


In [ ]:
class OrdinalRatingModel(nn.Module):
    def __init__(self, num_venues, venue_dim=3):
        super().__init__()

        self.scibert = AutoModel.from_pretrained(MODEL_NAME)

        # Original gradual unfreezing:
        # freeze bottom layers, unfreeze top 2 encoder layers + pooler.
        for name, param in self.scibert.named_parameters():
            if (
                "encoder.layer.10" in name
                or "encoder.layer.11" in name
                or "pooler" in name
            ):
                param.requires_grad = True
            else:
                param.requires_grad = False

        self.venue_embedding = nn.Embedding(num_venues, venue_dim)

        fusion_dim = 768 + 1 + venue_dim

        self.regressor = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1),
        )

        self.projection_head = nn.Linear(fusion_dim, 64)

    def forward(self, input_ids, attention_mask, year, venue):
        outputs = self.scibert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        venue_embedding = self.venue_embedding(venue)
        year = year.unsqueeze(1)

        features = torch.cat(
            (cls_embedding, year, venue_embedding),
            dim=1,
        )

        pred = self.regressor(features).squeeze(1)
        projection = self.projection_head(features)

        return pred, projection


## Cell 4 — 5-Fold Cross Validation Training

The notebook saves:

- best fold checkpoints
- OOF continuous predictions
- OOF discrete predictions after global threshold optimization


In [ ]:
skf = StratifiedKFold(
    n_splits=FOLDS,
    shuffle=True,
    random_state=SEED,
)

oof_predictions = np.zeros(len(train_df), dtype=np.float32)
oof_labels = np.zeros(len(train_df), dtype=np.float32)

print("=" * 60)
print("Training configuration")
print("=" * 60)
print("Method:", EXPERIMENT_NAME)
print("FOLDS:", FOLDS)
print("EPOCHS:", EPOCHS)
print("ALPHA:", ALPHA)
print("BATCH_SIZE:", BATCH_SIZE)
print("MAX_LEN:", MAX_LEN)
print("=" * 60)

for fold, (train_idx, val_idx) in enumerate(
    skf.split(train_df, train_df[LABEL_COL])
):
    print(f"\n{'=' * 20} FOLD {fold + 1}/{FOLDS} {'=' * 20}")

    seed_everything(SEED + fold)

    train_data = train_df.iloc[train_idx].reset_index(drop=True)
    val_data = train_df.iloc[val_idx].reset_index(drop=True)

    generator = torch.Generator()
    generator.manual_seed(SEED + fold)

    train_loader = DataLoader(
        PaperDataset(train_data, tokenizer, max_len=MAX_LEN),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        PaperDataset(val_data, tokenizer, max_len=MAX_LEN),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )

    model = OrdinalRatingModel(num_venues=num_venues).to(device)

    mse_criterion = nn.MSELoss()
    contrastive_criterion = OrdinalContrastiveLoss()

    scibert_params = [
        p for n, p in model.named_parameters()
        if "scibert" in n and p.requires_grad
    ]

    other_params = [
        p for n, p in model.named_parameters()
        if "scibert" not in n and p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        [
            {"params": scibert_params, "lr": 2e-5},
            {"params": other_params, "lr": 2e-4},
        ]
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
    )

    best_val_loss = float("inf")
    best_fold_preds = None
    best_model_path = MODEL_DIR / f"best_model_fold_{fold}.pt"

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0

        pbar = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{EPOCHS} [Train]"
        )

        for batch in pbar:
            optimizer.zero_grad()

            preds, projection = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["year"].to(device),
                batch["venue"].to(device),
            )

            labels = batch["label"].to(device)

            mse_loss = mse_criterion(preds, labels)
            contrastive_loss = contrastive_criterion(projection, labels)

            loss = mse_loss + ALPHA * contrastive_loss

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        model.eval()
        val_loss = 0.0
        val_preds_fold = []

        with torch.no_grad():
            for batch in tqdm(
                val_loader,
                desc=f"Epoch {epoch + 1}/{EPOCHS} [Val]",
                leave=False,
            ):
                preds, projection = model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["year"].to(device),
                    batch["venue"].to(device),
                )

                labels = batch["label"].to(device)

                mse_loss = mse_criterion(preds, labels)
                contrastive_loss = contrastive_criterion(projection, labels)

                loss = mse_loss + ALPHA * contrastive_loss

                val_loss += loss.item()
                val_preds_fold.extend(preds.cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)

        scheduler.step(avg_val_loss)

        print(
            f"Epoch {epoch + 1} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.2e}"
        )

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), best_model_path)
            best_fold_preds = np.array(val_preds_fold)

    oof_predictions[val_idx] = best_fold_preds
    oof_labels[val_idx] = val_data[LABEL_COL].values

    del model
    torch.cuda.empty_cache()


print("\nOptimizing global thresholds on OOF predictions...")
optR = OptimizedRounder()
best_global_thresholds = optR.fit(oof_predictions, oof_labels)
final_oof_discrete = optR.predict(oof_predictions, best_global_thresholds)

oof_qwk = cohen_kappa_score(
    oof_labels,
    final_oof_discrete,
    weights="quadratic",
)

print("Final thresholds:", best_global_thresholds)
print(f"Global OOF QWK Score: {oof_qwk:.6f}")

oof_df = pd.DataFrame({
    "id": train_df["id"].values,
    "y_true": oof_labels.astype(int),
    "pred_continuous": oof_predictions,
    "pred_label": final_oof_discrete.astype(int),
})

oof_path = SUBMISSION_DIR / "oof_scibert_contrastive_regression_cv5.csv"
oof_df.to_csv(oof_path, index=False)

print("Saved OOF predictions to:", oof_path)
oof_df.head()


## Cell 5 — Test Inference and Submission

This cell saves:

```text
submissions/submission.csv
submissions/submission_scibert_contrastive_regression_cv5.csv
submissions/test_pred_scibert_contrastive_regression_cv5.csv
```

The test prediction file includes continuous predictions for public and private test rows.  
It is useful for later weighted ensembling.


In [ ]:
public_test_df = pd.read_csv(DATA_DIR / "public_test.csv")
private_test_df = pd.read_csv(DATA_DIR / "private_test.csv")


def preprocess_test_data(test_df, min_y, max_y, v_map):
    df = test_df.copy()

    df["year_norm"] = (
        df["year"] - min_y
    ) / (max_y - min_y + 1e-8)

    df["venue_id"] = (
        df["venue"]
        .fillna("UNKNOWN")
        .map(lambda x: v_map.get(x, 0))
    )

    return df


public_processed = preprocess_test_data(
    public_test_df,
    min_year,
    max_year,
    venue_map,
)

private_processed = preprocess_test_data(
    private_test_df,
    min_year,
    max_year,
    venue_map,
)

public_loader = DataLoader(
    InferencePaperDataset(public_processed, tokenizer, max_len=MAX_LEN),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

private_loader = DataLoader(
    InferencePaperDataset(private_processed, tokenizer, max_len=MAX_LEN),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


@torch.no_grad()
def get_ensemble_continuous_predictions(loader, num_folds, device):
    ensemble_preds = np.zeros(len(loader.dataset), dtype=np.float32)

    for fold in range(num_folds):
        print(f"  -> Predicting Fold {fold}")

        fold_model = OrdinalRatingModel(num_venues=num_venues).to(device)

        model_path = MODEL_DIR / f"best_model_fold_{fold}.pt"

        fold_model.load_state_dict(
            torch.load(
                model_path,
                map_location=device,
            )
        )

        fold_model.eval()

        fold_preds = []

        for batch in loader:
            preds, _ = fold_model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["year"].to(device),
                batch["venue"].to(device),
            )

            fold_preds.extend(preds.cpu().numpy())

        ensemble_preds += np.array(fold_preds, dtype=np.float32) / num_folds

        del fold_model
        torch.cuda.empty_cache()

    return ensemble_preds


print("Running public test inference...")
public_continuous = get_ensemble_continuous_predictions(
    public_loader,
    FOLDS,
    device,
)

print("Running private test inference...")
private_continuous = get_ensemble_continuous_predictions(
    private_loader,
    FOLDS,
    device,
)

public_labels = optR.predict(
    public_continuous,
    best_global_thresholds,
).astype(int)

private_labels = optR.predict(
    private_continuous,
    best_global_thresholds,
).astype(int)

public_submission = pd.DataFrame({
    "id": public_test_df["id"].values,
    "Label": public_labels,
})

private_submission = pd.DataFrame({
    "id": private_test_df["id"].values,
    "Label": private_labels,
})

final_submission = pd.concat(
    [public_submission, private_submission],
    ignore_index=True,
)

submission_path = SUBMISSION_DIR / "submission.csv"
submission_exp_path = SUBMISSION_DIR / "submission_scibert_contrastive_regression_cv5.csv"

final_submission.to_csv(submission_path, index=False)
final_submission.to_csv(submission_exp_path, index=False)

test_pred_df = pd.concat(
    [
        pd.DataFrame({
            "id": public_test_df["id"].values,
            "split": "public",
            "pred_continuous": public_continuous,
            "pred_label": public_labels,
        }),
        pd.DataFrame({
            "id": private_test_df["id"].values,
            "split": "private",
            "pred_continuous": private_continuous,
            "pred_label": private_labels,
        }),
    ],
    ignore_index=True,
)

test_pred_path = SUBMISSION_DIR / "test_pred_scibert_contrastive_regression_cv5.csv"
test_pred_df.to_csv(test_pred_path, index=False)

print("Saved submission to:", submission_path)
print("Saved experiment submission to:", submission_exp_path)
print("Saved test continuous predictions to:", test_pred_path)
print("Submission shape:", final_submission.shape)
print("Submission distribution:")
print(final_submission["Label"].value_counts().sort_index())

final_submission.head()
